# Comparing pickle vs netcdf


In [1]:
%%time

import os
import datacube
import numpy as np
import pandas as pd
import xarray as xr
import datetime as dt
import matplotlib.pyplot as plt
import pickle
import time

CPU times: user 2.4 s, sys: 2.17 s, total: 4.57 s
Wall time: 16.5 s


In [2]:
# Check memory usage
import resource
def memory_usage():
    usage = resource.getrusage(resource.RUSAGE_SELF)
    return f"Memory usage: {usage.ru_maxrss / 1024} MB"
print(memory_usage())

Memory usage: 392.3515625 MB


In [16]:
stubs = ['MILGA2_ds2', 'fm_ndwi_5_ds2', 'ADAMO_ds2']
for stub in stubs:
    scratch_netcdf = f'/scratch/xe2/cb8590/tmp/{stub}.nc'
    gdata_netcdf = f'/g/data/xe2/Chris/Data/PadSeg/{stub}.nc'

    start = time.time()
    print(f"Loading from scratch: {stub}.nc")
    ds_netcdf = xr.open_dataset(scratch_netcdf)
    ds_netcdf.load()
    print(memory_usage())
    print("Time: ", time.time()-start, "seconds")
    print(ds_netcdf['nbart_blue'].shape)
    print()
    
    start = time.time()
    print(f"Loading from g/data: {stub}.nc")
    ds_netcdf = xr.open_dataset(gdata_netcdf)
    ds_netcdf.load()
    print(memory_usage())
    print("Time: ", time.time()-start, "seconds")
    print(ds_netcdf['nbart_blue'].shape)
    print()

Loading from scratch: MILGA2_ds2.nc
Memory usage: 36021.8125 MB
Time:  0.9231631755828857 seconds
(71, 424, 387)

Loading from g/data: MILGA2_ds2.nc
Memory usage: 36021.8125 MB
Time:  3.777817487716675 seconds
(71, 424, 387)

Loading from scratch: fm_ndwi_5_ds2.nc
Memory usage: 36021.8125 MB
Time:  2.545215368270874 seconds
(236, 424, 386)

Loading from g/data: fm_ndwi_5_ds2.nc
Memory usage: 36021.8125 MB
Time:  5.83074951171875 seconds
(236, 424, 386)

Loading from scratch: ADAMO_ds2.nc
Memory usage: 58342.26171875 MB
Time:  14.096793174743652 seconds
(236, 1069, 965)

Loading from g/data: ADAMO_ds2.nc
Memory usage: 58342.28125 MB
Time:  45.41155767440796 seconds
(236, 1069, 965)



In [3]:
stub = 'ADAMO_ds2'

john_padseg = '/g/data/xe2/John/Data/PadSeg/'
original_filepath = john_padseg + stub + '.pkl'

chris_temp = '/scratch/xe2/cb8590/tmp/'

outpath_pickle = f'/scratch/xe2/cb8590/tmp/{stub}.pkl'
outpath_netcdf = f'/scratch/xe2/cb8590/tmp/{stub}.nc'

# Comment out the cell below after running it. Then restart the kernel and run all cells.

In [4]:
# # # %%time

# # # Create two new files for testing
# with open(original_filepath, 'rb') as handle:
#     ds = pickle.load(handle)
    
# # with open(outpath_pickle, 'wb') as handle:
# #     pickle.dump(ds, handle, protocol=pickle.HIGHEST_PROTOCOL)

# # Sometimes need to remove these for the netcdf save to work
# ds['time'].attrs.pop('units', None)
# if 'flags_definition' in ds.attrs:
#     ds.attrs.pop('flags_definition')
# for var in ds.variables:
#     if 'flags_definition' in ds[var].attrs:
#         ds[var].attrs.pop('flags_definition')

# ds.to_netcdf(outpath_netcdf)

# print(memory_usage())

# Second time running this notebook (after restarting the kernel)

In [5]:
print(memory_usage())

Memory usage: 392.3515625 MB


In [6]:
%%time
# NetCDF Load

ds_netcdf = xr.open_dataset(outpath_netcdf)

print(memory_usage())

Memory usage: 392.3515625 MB
CPU times: user 130 ms, sys: 65.6 ms, total: 196 ms
Wall time: 601 ms


In [13]:
%%time
# Pickle load

with open(outpath_pickle, 'rb') as handle:
    ds_pickle = pickle.load(handle)
    
print(memory_usage())

Memory usage: 36019.01171875 MB
CPU times: user 10.5 ms, sys: 2.37 ms, total: 12.8 ms
Wall time: 31.8 ms


In [8]:
# Need to remove an existing .nc file before you can make a new one
if os.path.exists(outpath_netcdf):
    os.remove(outpath_netcdf)

In [14]:
%%time
# NetCDF Save

ds_netcdf.to_netcdf(outpath_netcdf)

print(memory_usage())

Memory usage: 36019.01171875 MB
CPU times: user 111 ms, sys: 1min 11s, total: 1min 11s
Wall time: 1min 33s


In [12]:
%%time
# Pickle save

with open(outpath_pickle, 'wb') as handle:
    pickle.dump(ds_netcdf, handle, protocol=pickle.HIGHEST_PROTOCOL)
    
print(memory_usage())

Memory usage: 36019.01171875 MB
CPU times: user 470 µs, sys: 12.3 ms, total: 12.7 ms
Wall time: 23.5 ms


In [ ]:
ds_pickle

In [ ]:
ds_netcdf

In [ ]:
!du -sh {outpath_pickle}

In [ ]:
!du -sh {outpath_netcdf}

In [ ]:
!du -sh {original_filepath}